utils file used to :
1- loading data 
2- definning functions 

In [0]:


# Databricks notebook source
# =============================================================================
# monitoring_utils_v3_final.py
# =============================================================================
# Shared helpers for the file-monitoring pipeline.
#
# Tasks:
#   A. Constants & mask registry
#   B. Delta Table helpers  (config + alarm log)
#   C. Date / window helpers
#   D. Filename helpers
#   E. File-system scanner
#   F. ADF pipeline client
#   G. Alarm-level logic
#
# Migration log — dict → Spark DataFrame
# ---------------------------------------
# load_usecase_config  now returns a Spark DataFrame (one row per file).
# New helpers added so every downstream step stays pure-Spark:
#
#   get_scalar_config(config_df)           → dict of usecase-level scalars
#   get_expected_files_df(config_df, t)    → DataFrame + resolved_filename col
#   list_raw_files_df(config_df)           → DataFrame of found filenames
#   write_alarm_records_df(missing_df,..)  → single-action batch Delta write
#==============================================
# Bug fixes applied in this version
# -----------------------------------
# [B1] load_usecase_config: `if not returned_df` always True for a DataFrame
#      → replaced with .count() == 0 check
# [D1] resolve_filename: `template` variable undefined when neither the
#      from/to range branch nor the MonthName branch executes
#      → initialise `template = file_template` at function start
# [E1] get_expected_files: checked for column "data_files" (never exists)
#      → now checks "file_template", "raw_data_path", "subdirectory"
# [E2] get_expected_files: `if len(missing) > 1` — off-by-one; a single
#      missing column was silently ignored
#      → corrected to `> 0`
# [E3] get_expected_files: loop accessed row["data_files"] (NameError) and
#      returned the raw template instead of the resolved filename
#      → loop now uses row["file_template"] and returns resolved filenames
# [E4] list_raw_files: signature still (base_path:str, data_files:dict)
#      → replaced with list_raw_files_df(config_df) that returns a DataFrame
# [O1] Orchestrator steps 5/7/8 used undefined `config` dict (NameError)
#      → use meta = get_scalar_config(config_df) for scalar access
# [O2] Orchestrator step 6: `found_files` never defined
#      → replaced with DataFrame anti-join
# [O3] Orchestrator step 8: per-file write loop replaced by
#      write_alarm_records_df (single Spark action)


In [0]:
%pip install azure-mgmt-datafactory azure-identity  tomlrt

In [ ]:
%sql
-- Corresponding DDL
-- truncate  table  monitoring.usecase_config;

CREATE TABLE IF NOT EXISTS monitoring.usecase_config (
usecase_name STRING NOT NULL,
frequency STRING NOT NULL, -- D / W / M
lag_days INT, -- delivery lag in days (D only)
deadline_rule INT, -- NULL=daily | weekday 0-6 | day-of-month 1-31
date_mask STRING NOT NULL, -- yyyyMMdd | yyyyMM | yyyy | MMdd
length_of_date INT,
file_frequency STRING,
file_ownership BOOLEAN,
sender_type STRING,
sftp_area_path STRING,
raw_data_path STRING NOT NULL,
pipeline_name STRING,
subdirectory STRING, -- "" for flat layout
file_template STRING NOT NULL, -- single pattern per row
updated_at TIMESTAMP DEFAULT current_timestamp()
) USING DELTA
TBLPROPERTIES (
'delta.feature.allowColumnDefaults' = 'supported',
'delta.enableChangeDataFeed' = 'true'
);


In [ ]:
%sql

-- commit 3: offset of file (lag_days)  : delay between pipeline scheduling and date of filename (not ingestion date !) involved in ingestion for specific instance/execution  of the pipeline


-- Example INSERTs :
INSERT INTO monitoring.usecase_config VALUES
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_1_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_2_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_3_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_1_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_2_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_3_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_2','D',1,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_2/','usecase_2','folder_usecase_2','file_1_usecase_2_from[yyyyMMdd]_to[yyyyMMdd].csv.gz',current_timestamp()),
('usecase_2','D',1,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_2/','usecase_2','folder_usecase_2','file_2_usecase_2_from[yyyyMMdd]_to[yyyyMMdd].csv.gz',current_timestamp()),
('usecase_2','D',1,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_2/','usecase_2','folder_usecase_2','file_3_usecase_2_from[yyyyMMdd]_to[yyyyMMdd].csv.gz',current_timestamp())


In [0]:
import re
import logging
from datetime import datetime, timedelta, date
import calendar
from typing import Union
from azure.identity import DefaultAzureCredential 
# from azure.mgmt.resource import ResourceManagementClient
# from azure.mgmt.datafactory import DataFactoryManagementClient
# from azure.mgmt.datafactory.models import *
from datetime import datetime, timedelta
import time
import tomlrt


In [ ]:
with open("pyproject.toml", "rb") as f:
    doc = tomlrt.load(f)

    main_file=doc.get("paths")["main_file"]
    utils_file=doc.get("paths")["utils_file"]
    ALARM_LEVELS=doc.get("ALARM_LEVELS")
    _MASK_TO_STRFTIME=doc.get("_MASK_TO_STRFTIME")
    _SUPPORTED_FREQUENCIES=doc.get("_SUPPORTED_FREQUENCIES")["freq"]
    _CONFIG_TABLE=doc.get("tables")["_CONFIG_TABLE"]
    _ALARM_TABLE=doc.get("tables")["_ALARM_TABLE"]

In [0]:

# ============================================================================ #
# B. DELTA TABLE HELPERS                                                        #
# ============================================================================ #

def load_usecase_config(usecase_name: str):
    """
    Load usecase configuration from the Delta Table.

    Returns a Spark **DataFrame** with one row per expected file
    (same schema as monitoring.usecase_config, filtered to usecase_name).

    Columns
    -------
    usecase_name, frequency, lag_days, deadline_rule, date_mask,
    length_of_date, file_frequency, file_ownership, sender_type,
    sftp_area_path, raw_data_path, pipeline_name,
    subdirectory, file_template, updated_at

    Use get_scalar_config(df) to extract usecase-level scalars (frequency,
    pipeline_name, …) as a plain dict for functions that need scalar values.

    Raises
    ------
    ValueError  : usecase not found or all rows have empty file_template.
    RuntimeError: unexpected error reading from Delta Table.
    """
    if not isinstance(usecase_name, str) or not usecase_name.strip():
        raise ValueError("usecase_name must be a non-empty string.")

    usecase_name = usecase_name.strip().lower().replace(" ", "_")

    try:
        config_df = (
            spark.table(_CONFIG_TABLE)                           # noqa: F821
            .filter(f"usecase_name = '{usecase_name}'")
            .filter("file_template IS NOT NULL AND trim(file_template) != ''")
        )
        # FIX [B1]: DataFrame is always truthy — must call .count() to detect empty
        row_count = config_df.count()
    except Exception as exc:
        raise RuntimeError(
            f"Failed to query '{_CONFIG_TABLE}' for usecase '{usecase_name}': {exc}"
        ) from exc

    if row_count == 0:
        raise ValueError(
            f"Usecase '{usecase_name}' not found in '{_CONFIG_TABLE}' "
            "(or all rows have an empty file_template). "
            "Add at least one row with a non-empty file_template before running."
        )

    log.info(
        "[load_usecase_config] Loaded %d row(s) for usecase '%s'.",
        row_count, usecase_name,
    )
    return config_df


In [0]:
def extract_date(file_info):
    name = file_info.name

    try:
        # range pattern → take latest (toYYYYMMDD)
        match = re.search(r'to(\d{8})', name)
        if match:
            return datetime.strptime(match.group(1), "%Y%m%d")
        # normal 8-digit date
        match = re.search(r'(\d{8})', name)
        if match:
            return datetime.strptime(match.group(1), "%Y%m%d")

        # fallback 6-digit (yyyyMM)
        match = re.search(r'(\d{6})', name)
        if match:
            return datetime.strptime(match.group(1), "%Y%m")

    except:
        pass

    return datetime.min

In [0]:
def pattern_to_regex(pattern: str) -> str:
    """
    file pattern -> regex
    """
    regex = re.escape(pattern)

    # placeholders
    regex = regex.replace(r"\[yyyyMMdd\]", r"\d{8}")
    regex = regex.replace(r"\[yyyyMM\]", r"\d{6}")   

    regex = regex.replace("yyyyMMdd", r"\d{8}")
    regex = regex.replace("yyyyMM", r"\d{6}")        

    # regex = regex.replace(r"t_yyyyMM\", r"\d{6}")
    regex = regex.replace(r"\[MonthName\]", r"[A-Za-z]+")
    regex = regex.replace(r"\[FromMonthNameToMonthName\]", r"[A-Za-z]+")

    return regex   # NOT strict (^$ removed)



In [0]:
# import datetime
""" DEFINE WINDOWING OF THE FILE 


Daily → yesterday’s file
Weekly → last expected weekday
Monthly → last expected month/day
"""
# @daily 
# Expected_date=datetime.now()-deltaday(2)
# @weekly
# -> currentweek -> number_of_day  __ comparing with __ last loaded file  day(usecase_info["deadline"])
# @yearly
# -> currentyear -> number_of_day  __ comparing with __ last loaded file  day(usecase_info["deadline"]

In [0]:
# from datetime import datetime, timedelta, date

# def get_expected_file(file_formal, datamask, frequency, target_weekday, lag_days, deadline=None, previous_month=True):
#     """
#     Returns the expected reference date for a file.

#     Parameters:
#         file_formal: kept from your original signature
#         datamask: optional strftime mask for formatting the result
#         frequency: 'D', 'W', or 'M'
#         target_weekday: int from 0 (Monday) to 6 (Sunday)
#         lag_days: delay in days for daily files
#         deadline: optional date/datetime used by your process
#         previous_month: for monthly files, choose previous month if True

#     Returns:
#         date object or formatted string if datamask is provided
#     """
#     today = datetime.now().date()

#     if frequency == "D":
#         expected_date = today - timedelta(days=lag_days)

#     elif frequency == "W":
#         if not 0 <= target_weekday <= 6:
#             raise ValueError("target_weekday must be between 0 (Monday) and 6 (Sunday)")

#         # Most recent occurrence of the target weekday
#         delta_days = (today.weekday() - target_weekday) % 7
#         expected_date = today - timedelta(days=delta_days)

#     elif frequency == "M":
#         year = today.year
#         month = today.month

#         if previous_month:
#             if month == 1:
#                 year -= 1
#                 month = 12
#             else:
#                 month -= 1

#         # Use first day of the selected month as the reference date
#         expected_date = date(year, month, 1)

#     else:
#         raise ValueError("Invalid frequency. Use 'D', 'W', or 'M'.")

#     if datamask:
#         return expected_date.strftime(datamask)

#     return expected_date

In [0]:
def get_deadline(
    sysdate: datetime,
    frequency: str,
    deadline_rule,
    lag_days: int
):
    """
    Compute deadline timestamp based on frequency and rule.

    Parameters:
    - sysdate: current datetime (already timezone aligned)
    - frequency: "D", "W", "M"
    - deadline_rule:
        - "daily" for D
        - int (0-6) for W (Monday=0)
        - int (1-31) for M
   )
    
    Returns:
    - deadline (datetime)
    """
 
    if(deadline_rule<0 or deadline_rule>31):
            raise ValueError("deadline_rule has an Invalid value (deadline_line )")

    today = sysdate.date()
   
  

    if frequency == "D" and deadline_rule == 0:
        expected_date = today - timedelta(days=lag_days)
        deadline_date = expected_date + timedelta(days=2)
        return deadline_date

    elif frequency == "W":
        target_weekday = deadline_rule
        today_weekday = today.weekday()

        if today_weekday >= target_weekday:
            deadline_date = today + timedelta(days=(7 - (today_weekday - target_weekday)))
        else:
            deadline_date = today + timedelta(days=(target_weekday - today_weekday))

        return deadline_date


    elif frequency == "M":
     target_day = deadline_rule

     last_day = calendar.monthrange(today.year, today.month)[1]
     target_day = min(target_day, last_day)

     if today.day < target_day:
        deadline_date = today.replace(day=target_day)
     else:
        next_month = today.month + 1 if today.month < 12 else 1
        year = today.year if today.month < 12 else today.year + 1

        last_day_next = calendar.monthrange(year, next_month)[1]
        target_day_next = min(deadline_rule, last_day_next)

        deadline_date = date(year, next_month, target_day_next)
     return deadline_date
    else:
        raise ValueError("Invalid frequency")

In [0]:
# def add_date_to_file(date:datetime,file:str,datamask:str,length_of_date:int):
#  """
#   it'll be used by get_deadline to create expected files based on dates , file formats
#  """
#  if datamask=="yyyyMMdd" and length_of_date==8:
#     formatted = date.strftime("%Y%m%d")
#     file.replace(datamask, formatted)
#  elif datamask=="yyyyMM" and length_of_date==6:
#     formatted = date.strftime("%Y%m")
#     file.replace(datamask, formatted)
#  elif datamask=="from[YYYYMMDD]_to[YYYYMMDD]" :
#    print("still in progress")





In [0]:
# def get_expected_file(file_formal, datamask, frequency, target_weekday, lag_days, deadline=None, previous_month=True):
# get_expected_file()

In [0]:
from datetime import datetime, timedelta, date
from typing import Union

# --------------------------------------------------------------------------- #
#  Mask registry – single source of truth for custom mask → strftime mapping  #
# --------------------------------------------------------------------------- #
_MASK_TO_STRFTIME = {
    "yyyyMMdd": "%Y%m%d",
    "yyyyMM":   "%Y%m",
    "yyyy":     "%Y",
    "MMdd":     "%m%d",
}


def get_expected_file(
    file_formal: str,
    datamask: str | None,
    frequency: str,
    target_weekday= 0,
    lag_days: int = 0,
    deadline=None,
    previous_month: bool = True,
) -> Union[date, str]:
    """
    Returns the expected reference date for a file, optionally formatted.

    Parameters
    ----------
    file_formal     : kept for downstream use (e.g. passed to add_date_to_file)
    datamask        : custom mask key ('yyyyMMdd', 'yyyyMM') or None for raw date
    frequency       : 'D' daily | 'W' weekly | 'M' monthly
    target_weekday  : 0=Monday … 6=Sunday  (weekly only)
    lag_days        : days to subtract from today  (daily only)
    deadline        : optional override date/datetime (not yet used internally)
    previous_month  : if True, monthly files reference the previous month
    """
    today = datetime.now().date()

    if frequency == "D":
        expected_date = today - timedelta(days=lag_days)

    elif frequency == "W":
        if not 0 <= target_weekday <= 6:
            raise ValueError("target_weekday must be 0 (Monday) … 6 (Sunday)")
        delta_days = (today.weekday() - target_weekday) % 7
        expected_date = today - timedelta(days=delta_days)

    elif frequency == "M":
        year, month = today.year, today.month
        if previous_month:
            month -= 1
            if month == 0:
                month = 12
                year -= 1
        expected_date = date(year, month, 1)

    else:
        raise ValueError(f"Unknown frequency '{frequency}'. Use 'D', 'W', or 'M'.")

    # if datamask:/
    date_formated=_format_date(expected_date, datamask)
    return add_date_to_file(expected_date, file_formal,datamask) # convert format of date according to datamask(yyyyMMdd,yyyyMM) (strftime)
    # return expected_date


def add_date_to_file(
    ref_date: Union[date, datetime],
    file: str,
    datamask: str,
) -> str:
    """
    Replaces the datamask placeholder inside a filename template with the
    formatted date and returns the resulting filename.

    Parameters
    ----------
    ref_date  : date or datetime – typically the output of get_expected_file()
    file      : filename template, e.g. 'sales_yyyyMMdd.csv'
    datamask  : placeholder to replace, e.g. 'yyyyMMdd' or 'yyyyMM'
                Also supports 'from[yyyyMMdd]_to[yyyyMMdd]' for weekly ranges.

    Returns
    -------
    str – filename with placeholder replaced by the actual date string.

    Examples
    --------
    >>> add_date_to_file(date(2025, 4, 14), "sales_yyyyMMdd.csv", "yyyyMMdd")
    'sales_20250414.csv'

    >>> add_date_to_file(date(2025, 4, 14), "report_from[yyyyMMdd]_to[yyyyMMdd].csv",
    ...                  "from[yyyyMMdd]_to[yyyyMMdd]")
    'report_from[20250410]_to[20250416].csv'
    """
    # --- range mask (weekly: Mon → Sun of ref_date's week) ---
    if datamask == "from[yyyyMMdd]_to[yyyyMMdd]":
        if isinstance(ref_date, datetime):
            ref_date = ref_date.date()
        week_start = ref_date - timedelta(days=ref_date.weekday())   # Monday
        week_end   = week_start + timedelta(days=6)                  # Sunday
        from_str   = week_start.strftime("%Y%m%d")
        to_str     = week_end.strftime("%Y%m%d")
        return file.replace("from[yyyyMMdd]", f"from[{from_str}]") \
                   .replace("to[yyyyMMdd]",   f"to[{to_str}]")

    # --- single-date masks ---
    formatted = _format_date(ref_date, datamask)   # raises if mask unknown
    return file.replace(datamask, formatted)        # ✅ result is now returned


# --------------------------------------------------------------------------- #
#  Internal helper                                                              #
# --------------------------------------------------------------------------- #
def _format_date(ref_date: Union[date, datetime], datamask: str) -> str:
    """Converts a date to string using the custom mask registry."""
    strftime_mask = _MASK_TO_STRFTIME.get(datamask)
    if not strftime_mask:
        raise ValueError(
            f"Unknown datamask '{datamask}'. "
            f"Supported: {list(_MASK_TO_STRFTIME)} + 'from[yyyyMMdd]_to[yyyyMMdd]'"
        )
    return ref_date.strftime(strftime_mask)

In [0]:

def add_date_to_file(
    ref_date: Union[date, datetime],
    file: str,
    datamask: str,
) -> str:
    """
    Replaces the datamask placeholder inside a filename template with the
    formatted date and returns the resulting filename.

    Parameters
    ----------
    ref_date  : date or datetime – typically the output of get_expected_file()
    file      : filename template, e.g. 'traffico_per_offerte_NG__yyyyMMdd.csv'
    datamask  : placeholder to replace, e.g. 'yyyyMMdd' or 'yyyyMM'
                Also supports 'from[yyyyMMdd]_to[yyyyMMdd]' for weekly ranges.

    Returns
    -------
    str – filename with placeholder replaced by the actual date string.

    Examples
    --------
    >>> add_date_to_file(date(2025, 4, 14), "traffico_per_offerte_NG__yyyyMMdd.csv", "yyyyMMdd")
    'traffico_per_offerte_NG__20250414.csv'
    >>> add_date_to_file(date(2025, 4), "traffico_per_yyyyMM.csv", "yyyyMM")
    'traffico_per_offerte_NG__202504.csv'

    >>> add_date_to_file(date(2025, 4, 14), "report_from[yyyyMMdd]_to[yyyyMMdd].csv",
    ...                  "from[yyyyMMdd]_to[yyyyMMdd]")
    'report_from[20250410]_to[20250416].csv'
    """
    # --- range mask (weekly: Mon → Sun of ref_date's week) ---
    if datamask == "from[yyyyMMdd]_to[yyyyMMdd]":
        if isinstance(ref_date, datetime):
            ref_date = ref_date.date()
        week_start = ref_date - timedelta(days=ref_date.weekday())   # Monday
        week_end   = week_start + timedelta(days=6)                  # Sunday
        from_str   = week_start.strftime("%Y%m%d")
        to_str     = week_end.strftime("%Y%m%d")
        return file.replace("from[yyyyMMdd]", f"from[{from_str}]") \
                   .replace("to[yyyyMMdd]",   f"to[{to_str}]")

    # --- single-date masks ---
    formatted = _format_date(ref_date, datamask)   # raises if mask unknown
    return file.replace(datamask, formatted)        # ✅ result is now returned



In [0]:
# get_deadline(datetime.now(), "M", 90,1)
# expected_file=get_expected_file(
#   file_formal="traffico_perNG_yyyyMM.csv",
#   datamask="yyyyMM",
#   frequency="M",
#   target_weekday=15,
#   previous_month=True)
# print(expected_file)
#  file_formal: str,
#     datamask: str | None,
#     frequency: str,
#     target_weekday: int = 0,
#     lag_days: int = 0,
#     deadline=None,
#     previous_month: bool = True,

    # file_formal: str,
    # datamask: str | None,
    # frequency: str,
    # target_weekday: int = 0,
    # lag_days: int = 0,
    # deadline=None,
    # previous_month: bool = True

In [0]:
# add_date_to_file(datetime.now,file_formal,_format_date(expected_date, datamask))
# def add_date_to_file(
#     ref_date: Union[date, datetime],
#     file: str,
#     datamask: str,

In [0]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.datafactory import DataFactoryManagementClient
from datetime import datetime, timedelta
import logging
from dotenv import load_dotenv

def retrieve_lastRun(pipeline_name:str,logging:bool,start_time,end_time):
    if logging:
        logger = logging.getLogger("azure.identity")
        logger.setLevel(logging.DEBUG)

        # Direct logging output to stdout. Without adding a handler,
        handler = logging.StreamHandler()
        logger.addHandler(handler)
    vars=load_dotenv()  # Load environment variables from .env file
    subscription_id = vars.get("subscription_id")

    # This program creates this resource group. If it's an existing resource group, comment out the code that creates the resource group
    resource_group = var.get("resource_group")

    # The data factory name. 
    factory_name = var.get("factory_name")
 

    # Authentication using DefaultAzureCredential (dev)
    credential = DefaultAzureCredential()
    client = DataFactoryManagementClient(credential, subscription_id)


    # Query pipeline runs
    runs_reponse = client.pipeline_runs.query_by_factory(
    resource_group,
    factory_name,
    filter_parameters={
        "filters":[{"operand":"PipelineName","operator":"In","values":[pipeline_name]}
                   ,{"operand":"LatestOnly","operator":"Equals","values":[True]}],"orderBy":[{"orderBy":"RunStart","order":"DESC"}],
        "last_updated_after": start_time,
        "last_updated_before": end_time,
    } 
    
    )
    return max(runs_reponse.value, key=lambda r: r.run_start)
    # max(runs_reponse.value, key=lambda r: r.run_start)
    




In [0]:
end_time = datetime.utcnow()
start_time = end_time - timedelta(days=1)
latest_run=retrieve_lastRun("erogazione_offerte",logging=False,start_time=start_time,end_time=end_time)

